# 02 — Data Cleaning: Merapikan Dataset

**Tujuan:** mengubah data mentah menjadi `songs_clean.csv` yang **konsisten dan
bisa dipecah per genre**, tanpa membuang informasi yang berguna.

Semua keputusan di notebook ini sudah disepakati di notebook 01. Prinsipnya:

> Rapikan yang jelas salah; **jangan buang data hanya karena terlihat aneh** —
> tandai saja kalau perlu.

Urutan pengerjaan (urutan penting!):

1. `explicit` dipastikan bertipe boolean
2. **Buang duplikat lebih dulu** (saat kolom masih sederhana, sebelum ada kolom bertipe list)
3. `genre`: `set()` → `"Unknown"`, lalu dipecah menjadi `genre_list` + `primary_genre`
4. `duration_ms` → `duration_min` (menit, lebih mudah dibaca manusia)
5. Tandai `popularity == 0` sebagai data tidak tersedia (`is_popularity_missing`)
6. Validasi dengan `assert`, lalu simpan ke `data/clean/songs_clean.csv`

## 1. Muat data mentah (read-only)

In [1]:
import numpy as np
import pandas as pd

RAW_PATH = "../data/raw/songs_normalize.csv"
CLEAN_PATH = "../data/clean/songs_clean.csv"

df = pd.read_csv(RAW_PATH)
baris_awal = len(df)
print("Baris data mentah:", baris_awal)

clean = df.copy()

Baris data mentah: 2000


## 2. Rapikan `explicit`

Pandas sudah membaca kolom ini sebagai boolean, tapi kita paksa normalisasi
secara eksplisit supaya notebook ini tetap aman kalau suatu hari datanya berubah
menjadi teks "True"/"False".

In [2]:
clean["explicit"] = (
    clean["explicit"].astype(str).str.strip().str.lower()
    .map({"true": True, "false": False})
)
# Fail fast: kalau ada nilai di luar true/false, hentikan di sini
assert clean["explicit"].isin([True, False]).all(), "Ada nilai explicit yang tidak valid!"
print("Jenis explicit:", clean["explicit"].unique())
print("Proporsi lagu explicit:", round(clean["explicit"].mean(), 3))

Jenis explicit: [False  True]
Proporsi lagu explicit: 0.276


## 3. Tangani duplikat (sebelum membuat kolom turunan)

Dua tahap, sesuai temuan notebook 01:

1. **Duplikat persis** → dibuang (sampah murni).
2. **Duplikat `artist`+`song` dengan metadata berbeda** → simpan satu baris dengan
   popularity **tertinggi** (seri: tahun paling awal). Kenapa? Karena baris dengan
   popularity 0 datanya kurang lengkap; versi popularity tertinggi adalah
   representasi lagu yang paling informatif.

Diurutkan dengan `kind="stable"` supaya hasilnya deterministik (selalu sama tiap
dijalankan). Tahap ini dilakukan **sebelum** membuat kolom bertipe list, karena
`duplicated()` pada DataFrame dengan kolom list akan error (list tidak bisa di-hash).

In [3]:
# Tahap 1: duplikat persis
n_dupe_persis = clean.duplicated().sum()
clean = clean.drop_duplicates()
print("Duplikat persis dibuang:", n_dupe_persis)

# Tahap 2: pasangan artist+song, simpan popularity tertinggi (seri: tahun paling awal)
n_dupe_lagu = clean.duplicated(subset=["artist", "song"]).sum()
clean = (
    clean.sort_values(["popularity", "year"], ascending=[False, True], kind="stable")
    .drop_duplicates(subset=["artist", "song"], keep="first")
)
print("Duplikat pasangan artist+song dibuang:", n_dupe_lagu)
print("Baris setelah deduplikasi:", len(clean), "dari", baris_awal)

Duplikat persis dibuang: 59
Duplikat pasangan artist+song dibuang: 15
Baris setelah deduplikasi: 1926 dari 2000


## 4. Normalisasi `genre` — perbaikan terpenting notebook ini

Tiga langkah:

1. `set()` (dan varian seperti list kosong / nan) → `"Unknown"` — menandakan genre tidak diketahui
2. Pecah string multi-genre menjadi list: `"pop, R&B"` → `["pop", "R&B"]`
3. Ambil genre pertama sebagai `primary_genre` untuk statistik **per lagu**

**Kenapa butuh `primary_genre`?** Kalau kita langsung meng-"explode" multi-genre,
satu lagu akan terhitung di banyak genre — total lagu jadi tidak konsisten (dobel
hitung). `primary_genre` memastikan **tiap lagu selalu terhitung tepat satu kali**.
Untuk analisis per-genre tetap ada tabel `genres_long` khusus (dibuat di notebook 03).

In [4]:
# 1. Kenali pola "genre tidak diketahui" — bukan hanya set(), tapi juga list kosong / nan
PATTERN_TIDAK_DIKETAHUI = r"^(set\(\)|nan|unknown|none)$"

clean["genre"] = clean["genre"].astype(str).str.strip()
mask_tidak_diketahui = clean["genre"].str.lower().str.match(PATTERN_TIDAK_DIKETAHUI)
print("Jumlah genre 'tidak diketahui' ditemukan:", mask_tidak_diketahui.sum())
clean.loc[mask_tidak_diketahui, "genre"] = "Unknown"

# 2. Pecah jadi list genre; 3. Ambil genre pertama sebagai primary_genre
clean["genre_list"] = clean["genre"].apply(
    lambda g: [bagian.strip() for bagian in g.split(",") if bagian.strip()]
)
clean["primary_genre"] = clean["genre_list"].apply(lambda daftar: daftar[0] if daftar else "Unknown")
clean["n_genres"] = clean["genre_list"].apply(len)

print("\nContoh hasil normalisasi:")
print(clean[["genre", "primary_genre", "n_genres"]].head(8).to_string())

Jumlah genre 'tidak diketahui' ditemukan: 22

Contoh hasil normalisasi:
                           genre primary_genre  n_genres
1322                   rock, pop          rock         2
1311                         pop           pop         1
201                      hip hop       hip hop         1
6                        hip hop       hip hop         1
1613  pop, R&B, Dance/Electronic           pop         3
1819       pop, Dance/Electronic           pop         2
209                      hip hop       hip hop         1
1227                         pop           pop         1


## 5. `duration_ms` → `duration_min`

Milidetik angka besar yang sulit dicerna (211160). Dalam menit jadi 3.52 — jauh lebih
mudah dibaca di grafik dan dashboard. Kolom asli tetap disimpan sebagai jejak.

In [5]:
clean["duration_min"] = (clean["duration_ms"] / 60_000).round(2)
print("Durasi (menit): min", clean["duration_min"].min(), "| median", clean["duration_min"].median(), "| max", clean["duration_min"].max())

Durasi (menit): min 1.88 | median 3.72 | max 8.07


## 6. Tandai `popularity == 0` sebagai data tidak tersedia

Bukan missing value (null), tapi secara makna "tidak ada data". Kita tandai dengan
flag supaya analisis bisa memilih: pakai semua lagu, atau kecualikan yang tanpa data.
**Tidak ada imputasi** — menebak popularity sama dengan membuat data palsu.

In [6]:
clean["is_popularity_missing"] = clean["popularity"] == 0
print("Lagu dengan popularity tidak tersedia:", clean["is_popularity_missing"].sum())
print("Rata-rata popularity (hanya yang ada datanya):",
      round(clean.loc[~clean["is_popularity_missing"], "popularity"].mean(), 1))

Lagu dengan popularity tidak tersedia: 121
Rata-rata popularity (hanya yang ada datanya): 63.8


## 7. Validasi akhir dengan assertion

Semua yang kita pastikan di notebook 01 dikunci di sini sebagai `assert`. Kalau suatu
hari data mentah berubah dan melanggar aturan, notebook akan **gagal di sini** —
bukan diam-diam menghasilkan analisis salah.

In [7]:
fitur_audio = ["danceability", "energy", "speechiness", "acousticness",
               "instrumentalness", "liveness", "valence"]

# Rentang & tipe
assert clean[fitur_audio].apply(lambda s: s.between(0, 1).all()).all(), "Fitur audio di luar 0-1"
assert clean["key"].between(0, 11).all(), "key di luar 0-11"
assert clean["mode"].isin([0, 1]).all(), "mode bukan 0/1"
assert (clean["tempo"] > 0).all(), "ada tempo tidak positif"
assert clean["explicit"].isin([True, False]).all(), "explicit bukan boolean"

# Hasil cleaning
assert not clean["genre"].str.lower().str.contains("set").any(), "masih ada set() di genre"
assert (clean["n_genres"] >= 1).all(), "ada lagu tanpa genre sama sekali"
assert clean["duration_min"].notna().all(), "ada duration_min kosong"
assert not clean.duplicated(subset=["artist", "song"]).any(), "masih ada duplikat lagu"
assert clean.isna().sum().sum() == 0, "ada null tersisa"

print("Semua validasi LULUS OK")

Semua validasi LULUS OK


## 8. Simpan hasil cleaning

Disimpan ke `data/clean/songs_clean.csv` (folder `clean` = kontrak antara notebook
dan dashboard). Notebook 03 akan membaca berkas ini sebagai bahan feature engineering.

In [8]:
clean = clean.reset_index(drop=True)
clean.to_csv(CLEAN_PATH, index=False)

print("Disimpan ke", CLEAN_PATH)
print("Bentuk akhir:", clean.shape)
print("\nKolom:", list(clean.columns))

# Ringkasan log cleaning
print(f"""
LOG CLEANING
------------
Baris awal                : {baris_awal}
Duplikat persis dibuang   : {n_dupe_persis}
Duplikat lagu dibuang     : {n_dupe_lagu}
Baris akhir               : {len(clean)}
Genre 'Unknown'           : {(clean['primary_genre'] == 'Unknown').sum()}
Popularity tidak tersedia : {clean['is_popularity_missing'].sum()}
""")

Disimpan ke ../data/clean/songs_clean.csv
Bentuk akhir: (1926, 23)

Kolom: ['artist', 'song', 'duration_ms', 'explicit', 'year', 'popularity', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'genre', 'genre_list', 'primary_genre', 'n_genres', 'duration_min', 'is_popularity_missing']

LOG CLEANING
------------
Baris awal                : 2000
Duplikat persis dibuang   : 59
Duplikat lagu dibuang     : 15
Baris akhir               : 1926
Genre 'Unknown'           : 22
Popularity tidak tersedia : 121



## 9. Ringkasan & langkah berikutnya

**Yang berubah dari data mentah:**

| Masalah | Penanganan | Hasil |
|---|---|---|
| `set()` di `genre` | → `"Unknown"` | 22 lagu ditandai jelas |
| Multi-genre | `genre_list` + `primary_genre` | bisa dicek per lagu **dan** per genre |
| `duration_ms` | → `duration_min` | lebih mudah dibaca |
| 59 duplikat persis | dibuang | data ringkas |
| 15 lagu kembar (metadata beda) | simpan popularity tertinggi | 1 lagu = 1 baris |
| `popularity = 0` | flag `is_popularity_missing` | tidak ada imputasi palsu |
| Tidak ada null | validasi `assert` | semua kriteria lulus |

**Berikutnya (notebook 03):** membuat fitur turunan — `key_name`, `mode_name`,
`decade`, `mood` (kuadran valence × energy), banding durasi/tempo/energi, penanda
outlier, plus tabel `artists` dan `genres_long` — lalu **menyimpan CSV + joblib**. Data
inilah yang dipakai dashboard.

## 10. Cerita tahap ini: "2.000 jadi 1.926 — dan itu justru bagus"

**Cerita tahap ini:**

> *Hook:* "Alessia Cara punya **dua** lagu 'Here' di dataset ini — satu tanpa data
> popularity sama sekali."
>
> *Angka:* "Dari 2.000 baris, 59 dibuang karena kembar persis, 15 lagu kembar
> lainnya disatukan (yang popularity-nya paling tinggi), 22 lagu kini jujur
> bergenre 'Unknown', dan 121 lagu popularity 0-nya ditandai sebagai 'tidak ada data'."
>
> *Makna:* "Cleaning bukan mengurangi data — cleaning membuat **setiap angka di
> dashboard bisa dipertanggungjawabkan**. 1.926 lagu yang bersih jauh lebih berguna
> daripada 2.000 baris yang campur aduk."

**Pola bercerita untuk tahap cleaning:** selalu pakai format
**sebelum → sesudah → kenapa**. Jangan pernah bilik "data sudah dibersihkan" tanpa
angka — pembaca (atau bosmu) akan langsung bertanya "berapa yang dibuang, dan kenapa?".

**Kebiasaan baik lain yang dicontohkan di notebook ini:**

- Setiap perubahan ditulis **apa yang dikerjakan + mengapa** (lihat §3, §5, §6).
- Tidak menghapus data yang "aneh" — cukup **ditandai** (`set()` → `Unknown`,
  popularity 0 → flag, outlier → `is_outlier`). Penghapusan hanya untuk duplikat.
- Semua aturan dikunci dengan `assert`, jadi kalau data ganti, notebook gagal
  **lebih dulu** ketimbang menghasilkan analisis salah.